# `RunnableBranch: RunnableSerializable[Input, Output]`

`RunnableBranch` selects and executes one `Runnable` according to ordered conditions.

It evaluates conditions from first to last. The first condition returning `True` selects its associated branch. When no condition matches, the default branch is executed.

## Type Parameters

```python
Input # Input type received by every condition and branch
Output # Output type produced by the selected branch
```

## Fields

```python
branches: Sequence[tuple[Runnable[Input, bool], Runnable[Input, Output]]] # Ordered condition-and-branch pairs
default: Runnable[Input, Output] # Runnable executed when no condition matches
```

## Constructor

```python
RunnableBranch(
    *branches: tuple[
        Runnable[Input, bool]
        | Callable[[Input], bool]
        | Callable[[Input], Awaitable[bool]],
        RunnableLike[Input, Output],
    ]
    | RunnableLike[Input, Output], # Condition-branch pairs followed by one default branch
) -> None # Initialize the conditional Runnable
```

The final positional argument is always treated as the default branch.

Conditions, branch actions, and the default action are automatically converted into `Runnable` objects when possible.

At least one condition-branch pair and one default branch must be supplied.

## Constructor Errors

- Raises `ValueError` when fewer than two positional arguments are supplied.
- Raises `TypeError` when the default branch is not a Runnable, callable, or mapping.
- Raises `TypeError` when a conditional branch is not a tuple or list.
- Raises `ValueError` when a conditional branch does not contain exactly two values.

## Branch Selection

```text
condition 1 → True → execute branch 1
            False
              ↓
condition 2 → True → execute branch 2
            False
              ↓
         execute default branch
```

Only the selected branch is executed.

Conditions after the first successful condition are not evaluated.

## Overridden Properties and Methods

### `is_lc_serializable`

Returns `True`, indicating that `RunnableBranch` supports LangChain serialization.

### `get_lc_namespace`

Returns the LangChain serialization namespace for Runnable objects.

### `get_input_schema`

Returns the first usable input schema found among the default branch, conditional branch actions, and conditions.

Falls back to the inherited Runnable input schema when none provides a usable schema.

### `config_specs`

Returns the unique configurable-field specifications collected from all conditions, conditional branch actions, and the default branch.

### `invoke`

Synchronously evaluates each condition in order and invokes the first matching branch.

Invokes the default branch when no condition returns `True`.

Additional keyword arguments are passed only to the selected branch.

### `ainvoke`

Asynchronously evaluates each condition in order and asynchronously invokes the first matching branch.

Asynchronously invokes the default branch when no condition returns `True`.

Additional keyword arguments are passed only to the selected branch.

### `stream`

Synchronously evaluates the conditions and streams chunks from the selected branch.

Streams from the default branch when no condition matches.

### `astream`

Asynchronously evaluates the conditions and asynchronously streams chunks from the selected branch.

Asynchronously streams from the default branch when no condition matches.

## Configuration and Tracing Behaviour

- The same input is passed to every evaluated condition and to the selected branch.
- Each condition receives its own child callback tagged as `condition:n`.
- Each selected conditional branch receives a child callback tagged as `branch:n`.
- The default branch receives a child callback tagged as `branch:default`.
- Runtime configuration is forwarded to conditions and the selected branch.
- Errors from conditions or branches are reported through the Runnable callback lifecycle and then raised.

## Developer-Facing Top-Level Statements

```python
RunnableBranch # Public conditional Runnable class
```

No public top-level functions or aliases are defined in this module.

In [ ]:
from langchain_core.runnables import RunnableBranch, RunnableLambda # Import required Runnable classes

def is_positive(number: int) -> bool: # Check whether the number is positive
    return number > 0 # Return True for positive numbers

def is_negative(number: int) -> bool: # Check whether the number is negative
    return number < 0 # Return True for negative numbers

def positive_message(number: int) -> str: # Define the positive-number branch
    return f"{number} is positive" # Return the positive result

def negative_message(number: int) -> str: # Define the negative-number branch
    return f"{number} is negative" # Return the negative result

def zero_message(number: int) -> str: # Define the default branch
    return f"{number} is zero" # Return the default result

number_branch = RunnableBranch( # Create the conditional Runnable
    (is_positive, RunnableLambda(positive_message)), # Execute when the number is positive
    (is_negative, RunnableLambda(negative_message)), # Execute when the number is negative
    RunnableLambda(zero_message), # Execute when no condition matches
) # Finish creating RunnableBranch

positive_result = number_branch.invoke(10) # Execute the positive branch

negative_result = number_branch.invoke(-5) # Execute the negative branch

zero_result = number_branch.invoke(0) # Execute the default branch

print(positive_result) # Display the positive result

print(negative_result) # Display the negative result

print(zero_result) # Display the default result